### SECTION09.로지스틱 회귀

주어진 데이터를 사용하여 데이터를 앞 50%와 나머지 50%로 나누십시오. a데이터로 로지스틱 회귀 모델을 적합하고, 다음 문제에 답하시오.

In [36]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import logit
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv('customer_travel.csv')
df.head()

,age,service,social,booked,target
0,34,6,0,1,0
1,34,5,1,0,1
2,37,3,1,0,0
3,30,2,0,0,0
4,30,1,0,0,0


0.데이터 나누기

In [5]:
point = len(df)//2
train = df.iloc[:point]
test = df.iloc[point:]

print("train :", train.shape)
print("test :", test.shape)

train : (400, 5)
test : (400, 5)


1.종속변수 target과 모든 독립변수를 사용하여 로지스틱 회귀 모델을 적합하고, 유의하지 않은 독립변수의 개수를 구하시오.(유의수준 0.05)

In [23]:
model = logit('target ~ age + service + social + booked',data=train).fit()
cond = model.pvalues[1:] >= 0.05
print("유의하지 않은 변수 개수 :", sum(cond))

Optimization terminated successfully.
         Current function value: 0.527521
         Iterations 6
유의하지 않은 변수 개수 : 2


2.p-value가 0.05보다 작은 유의한 변수만 사용하여 수정된 모델을 만들고 적합하시오. 이 수정된 모델에서 가장 큰 p-value를 가진 변수의 이름을 구하시오. 

In [26]:
model = logit('target ~ age + booked',data=train).fit()

col = model.pvalues[1:].idxmax()
pvalue = model.pvalues[1:].max()

print(col,":",pvalue)

Optimization terminated successfully.
         Current function value: 0.528581
         Iterations 6
age : 0.005922360830080401


3.수정된 모델에서 독립변수 중 절대값이 가장 큰 회귀계수를 가진 변수의 이름을 구하시오.

In [28]:
col = model.params[1:].abs().idxmax()
pvalue = model.params[1:].abs().max()
print(col,":",pvalue)

booked : 0.9460541604185365


In [29]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 target   No. Observations:                  400
Model:                          Logit   Df Residuals:                      397
Method:                           MLE   Df Model:                            2
Date:                Sun, 15 Nov 2026   Pseudo R-squ.:                 0.05064
Time:                        21:30:18   Log-Likelihood:                -211.43
converged:                       True   LL-Null:                       -222.71
Covariance Type:            nonrobust   LLR p-value:                 1.265e-05
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      2.4581      1.184      2.076      0.038       0.137       4.779
age           -0.1025      0.037     -2.752      0.006      -0.176      -0.030
booked        -0.9461      0.267     -3.542      0.000      -1.470      -0.423
==============================================================================
"""

4.수정된 모델에서 로그우도를 구하시오.

In [30]:
llf = model.llf
print("로그우도 :", llf)

로그우도 : -211.4323825144558


5.수정된 모델에서 잔차이탈도를 구하시오.

In [31]:
print("잔차이탈도 :",-2 * llf)

잔차이탈도 : 422.8647650289116


6.'booked'변수가 3증가했을 때의 오즈비를 구하시오.

In [33]:
val = np.exp(model.params['booked'] * 3)
print("booked 3증가 오즈비 :", val)

booked 3증가 오즈비 : 0.058533122917711476


7.p-value가 0.05보다 작은 회귀계수의 총합을 구하시오.(단 절편도 유의할 경우 포함)

In [35]:
cond = model.pvalues < 0.05
print("총합 :",model.params[cond].sum())

총합 : 1.4094682705861987


8.정확도를 측정하시오.

In [37]:
pred = model.predict(test)
pred = (pred > 0.5).astype(int)

accuracy = accuracy_score(test['target'], pred)
print("정확도 :", accuracy)

정확도 : 0.765


9.오류율을 계산하시오.

In [39]:
error_rate = 1-accuracy
print("오류율 :", error_rate)

오류율 : 0.235
